# Conflict Correlation Analysis

Perform statistical correlation analysis and hypothesis testing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 6)})

## 1. Load Data

In [ ]:
df = pd.read_parquet('outputs/processed/ais_features.parquet')
print(f'Records: {len(df):,}')

# Check if conflict labels exist
if 'conflict_label' in df.columns:
    print(f"Conflict records: {df['conflict_label'].sum():,}")
    print(f"Conflict zones: {df['conflict_zone_name'].unique()}")

## 2. Run Correlation Analyzer

In [ ]:
from src.analysis.correlation_analyzer import CorrelationAnalyzer

analyzer = CorrelationAnalyzer(output_dir='outputs/tables')

# Run all tests
results = analyzer.run_all_tests(
    df,
    features=['traffic_count', 'dark_ship_ratio', 'mean_rot_abs'],
    onset_date='2022-02-24',
    treatment_zone='black_sea'
)
print(f'Completed {len(results)} tests')

## 3. View Results

In [ ]:
results_df = pd.read_csv('outputs/tables/correlation_analysis_results.csv')
results_df

## 4. Hypothesis Testing Summary

In [ ]:
# H1: Traffic volume declines post-conflict
print('H1: Traffic volume declines post-conflict')
h1 = results_df[results_df['test'] == 'did']
if not h1.empty:
    for _, row in h1.iterrows():
        sig = '✓' if row.get('significant', False) else '✗'
        print(f"  {sig} Level change: {row.get('level_change', 0):.4f}, p={row.get('level_p_value', 1):.4f}")
else:
    print('  No DiD results available (need conflict event data)')

In [ ]:
# H2: Dark ship ratio increases (Granger lead)
print('H2: Dark ship ratio Granger leads conflict')
h2 = results_df[results_df['test'] == 'granger']
if not h2.empty:
    for _, row in h2.iterrows():
        sig = '✓' if row.get('significant', False) else '✗'
        print(f"  {sig} Feature: {row.get('feature')}, p={row.get('p_value', 1):.4f}, lag={row.get('best_lag', 0)}")
else:
    print('  No Granger results available')

## 5. Additional Analysis

In [ ]:
from src.analysis.traffic_analyzer import TrafficAnalyzer
from src.analysis.behavioral_analyzer import BehavioralAnalyzer

ta = TrafficAnalyzer(output_dir='outputs/tables')
ta.daily_traffic_volume(df)
ta.traffic_by_zone(df)

ba = BehavioralAnalyzer(output_dir='outputs/tables')
ba.route_entropy_analysis(df)
ba.loitering_analysis(df)
print('Additional analysis saved.')